# Climate Hazards — All CA Carceral Facilities

Joins all tract-level and facility-level climate hazard data to the 357-facility base dataset.

**Output:** `data/allfacilities_climate_hazards.csv`

## Hazard layers joined

| Layer | Join type | Source file |
|---|---|---|
| Heat & AQI | tract centroid → `tract_geoid` | `data/hazards/heat_air_hazard.csv` |
| Flood | tract centroid → `tract_geoid` | `data/hazards/flood_hazard.csv` |
| Drought | tract centroid → `tract_geoid` | `data/hazards/drought_hazard.csv` |
| Wildfire FHSZ | point-in-polygon | `data_sources/hazards/wildfire/calfire_fhsz.geojson` |
| Wildland-Urban Interface | point-in-polygon | `data_sources/hazards/wildfire/Wildland_Urban_Interface.zip` |

**Note on drought:** Run `data_sources/hazards/drought/drought_hazard.ipynb` first to generate `data/hazards/drought_hazard.csv`.

**Not included in this file (CDCR state prisons only):**
- Indoor/outdoor heat model: `data/cdcr/indoor_outdoor_heat_2025.csv`
- Heat activation days: `data_sources/hazards/heat/heat_activations_*.csv`
- Heat risk index: `data/cdcr/CDCR_heat_risk_index.csv`

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Base facilities
fac = pd.read_csv('data_sources/facilities/ca_facilities.csv')
print(f'Base facilities: {len(fac)}')
print(f'Types: {fac["type"].value_counts().to_dict()}')

# Normalize tract_geoid to zero-padded 11-digit string
fac['tract_str'] = fac['tract_geoid'].astype(str).str.split('.').str[0].str.zfill(11)

Base facilities: 357
Types: {'COUNTY': 188, 'STATE': 84, 'LOCAL': 54, 'FEDERAL': 28, 'MULTI': 3}


## 1. Tract-level hazards — heat, flood, drought

In [2]:
# ── Heat & AQI ──────────────────────────────────────────────────────────────
heat = pd.read_csv('data/hazards/heat_air_hazard.csv', dtype={'GEOID': str})
heat['tract_str'] = heat['GEOID'].str.zfill(11)

df = fac.merge(
    heat.drop(columns='GEOID'),
    on='tract_str', how='left'
)
print(f'After heat join: {df["heat_hazard_idx_norm"].notna().sum()} of {len(df)} facilities matched')

# ── Flood ────────────────────────────────────────────────────────────────────
flood = pd.read_csv('data/hazards/flood_hazard.csv', dtype={'GEOID': str})
flood['tract_str'] = flood['GEOID'].str.zfill(11)

df = df.merge(
    flood.drop(columns='GEOID'),
    on='tract_str', how='left'
)
print(f'After flood join: {df["flood_hazard_idx_norm"].notna().sum()} of {len(df)} facilities matched')

# ── Drought ──────────────────────────────────────────────────────────────────
import os
if os.path.exists('data/hazards/drought_hazard.csv'):
    drought = pd.read_csv('data/hazards/drought_hazard.csv', dtype={'GEOID': str})
    drought['tract_str'] = drought['GEOID'].str.zfill(11)
    df = df.merge(
        drought.drop(columns='GEOID'),
        on='tract_str', how='left'
    )
    print(f'After drought join: {df["drought_hazard_idx_norm"].notna().sum()} of {len(df)} facilities matched')
else:
    print('WARNING: data/hazards/drought_hazard.csv not found — run data_sources/hazards/drought/drought_hazard.ipynb first')
    print('Drought columns will be absent from output.')

After heat join: 357 of 357 facilities matched
After flood join: 357 of 357 facilities matched
After drought join: 357 of 357 facilities matched


## 2. Wildfire FHSZ — point-in-polygon

Fire Hazard Severity Zone classification (SRA + LRA layers combined).
Facilities outside all classified zones receive blank values (not classified = lowest risk).
`NonWildland` zones in the FHSZ layer are treated as unclassified.

In [3]:
# Build facility GeoDataFrame (WGS84)
fac_geo = gpd.GeoDataFrame(
    df[['facilityid', 'longitude', 'latitude']].copy(),
    geometry=[Point(xy) for xy in zip(df['longitude'], df['latitude'])],
    crs='EPSG:4326'
)

# Load FHSZ layer
fhsz = gpd.read_file('data_sources/hazards/wildfire/calfire_fhsz.geojson')
if fhsz.crs is None:
    fhsz = fhsz.set_crs('EPSG:4326')
fhsz = fhsz.to_crs('EPSG:4326')

# Point-in-polygon
fac_fhsz = gpd.sjoin(
    fac_geo, fhsz[['fhsz', 'responsibility', 'geometry']],
    how='left', predicate='within'
)

# Handle multiple matches (keep highest severity if a point lands in overlapping zones)
severity_order = {'Very High': 3, 'High': 2, 'Moderate': 1, 'NonWildland': 0}
fac_fhsz['_sev'] = fac_fhsz['fhsz'].map(severity_order).fillna(-1)
fac_fhsz = (
    fac_fhsz.sort_values('_sev', ascending=False)
    .drop_duplicates('facilityid')
    .drop(columns=['_sev', 'index_right'])
)

# Replace NonWildland with blank (not classified)
fac_fhsz['fhsz'] = fac_fhsz['fhsz'].replace('NonWildland', '')
fac_fhsz = fac_fhsz.rename(columns={'responsibility': 'fhsz_responsibility'})

# Clear responsibility for unclassified facilities
fac_fhsz.loc[fac_fhsz['fhsz'] == '', 'fhsz_responsibility'] = ''

df = df.merge(
    fac_fhsz[['facilityid', 'fhsz', 'fhsz_responsibility']],
    on='facilityid', how='left'
)
df['fhsz'] = df['fhsz'].fillna('')
df['fhsz_responsibility'] = df['fhsz_responsibility'].fillna('')

print(f'FHSZ classification summary:')
print(df['fhsz'].value_counts(dropna=False).to_string())
print(f'\nResponsibility area summary:')
print(df['fhsz_responsibility'].value_counts(dropna=False).to_string())

FHSZ classification summary:
fhsz
             248
Very High     62
Moderate      25
High          22

Responsibility area summary:
fhsz_responsibility
       248
SRA     66
LRA     43


## 3. Wildland-Urban Interface — point-in-polygon

WUI type from CalFire WUI boundaries (EPSG:3310). Facilities outside all WUI boundaries receive a blank value.

In [4]:
# Load WUI shapefile (EPSG:3310) and reproject to WGS84
wui = gpd.read_file('data_sources/hazards/wildfire/Wildland_Urban_Interface.zip')
wui = wui.to_crs('EPSG:4326')

# Reproject facility points to match
fac_geo_wgs = fac_geo.to_crs('EPSG:4326')

# Point-in-polygon
fac_wui = gpd.sjoin(
    fac_geo_wgs, wui[['WUI_DESC', 'geometry']],
    how='left', predicate='within'
)

# Keep one row per facility — WUI_DESC priority: Intermix > Interface > Influence Zone
wui_order = {'Intermix': 3, 'Interface': 2, 'Influence Zone': 1}
fac_wui['_wui_rank'] = fac_wui['WUI_DESC'].map(wui_order).fillna(0)
fac_wui = (
    fac_wui.sort_values('_wui_rank', ascending=False)
    .drop_duplicates('facilityid')
    .drop(columns=['_wui_rank', 'index_right'])
)

fac_wui = fac_wui.rename(columns={'WUI_DESC': 'wui_type'})

df = df.merge(
    fac_wui[['facilityid', 'wui_type']],
    on='facilityid', how='left'
)
df['wui_type'] = df['wui_type'].fillna('')

print('WUI type summary:')
print(df['wui_type'].value_counts(dropna=False).to_string())

WUI type summary:
wui_type
                  264
Influence Zone     64
Interface          16
Intermix           13


## 4. Clean up and rename columns, then output

**Dropped:** `benz_uhi_dt`, `benz_uhi_source` (methodology detail, documented in hazards README), `Dr_WSV_average`, `Dr_precip_demand_ratio` (opaque VCP sub-components captured by the drought index; precip/demand ratio has range 0.997–2.0 and requires DWR watershed knowledge to interpret).

**Scale convention:** composite indices are 0–100 (`_idx` suffix). Raw sub-components stay in natural units (days, °C, %). `heat_uhi_normalized` stays 0–1 (may slightly exceed 1 for non-CDCR facilities normalized against the CDCR max of 7.247°C).

**Prefix convention:** `heat_`, `flood_`, `drought_`, `fire_`.

In [5]:
# Drop methodology/opaque sub-component columns
drop_cols = [
    'benz_uhi_dt', 'benz_uhi_source',   # methodology detail
    'Dr_WSV_average', 'Dr_precip_demand_ratio',  # opaque VCP sub-components, captured by index
    'PollutionP',                         # all-pollution CalEnviroScreen percentile, off-topic here
    'flood_verywet_pre_pct', 'flood_verywet_fut_pct',  # hard to contextualize; floodplain % is clearer
    'tract_str',
]
output = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Rename for clarity:
#   - hazard prefix (heat_ / flood_ / drought_ / fire_)
#   - temporal suffixes: _historic (current period) and _midcentury (2041–2070)
#   - composite indices: 0–100, _idx suffix (no _norm)
#   - raw sub-components in natural units
rename_map = {
    # UHI (heat exposure, 0–1 normalized against CDCR max)
    'uhi_normalized':              'heat_uhi_normalized',
    # Heat sub-components
    'days_over_90_historic':       'heat_days_over_90_historic',
    'days_over_90_midcentury':     'heat_days_over_90_midcentury',
    'delta_90':                    'heat_days_over_90_delta',
    'hotnights_pre_pct':           'heat_hotnights_historic_pct',
    'hotnights_fut_pct':           'heat_hotnights_midcentury_pct',
    'AQI_norm':                    'heat_aqi_pctile',
    # Heat composite indices (0–100)
    'heat_hazard_idx_norm':        'heat_hazard_historic_idx',
    'heat_hazard_fut_idx_norm':    'heat_hazard_midcentury_idx',
    # Flood composite indices (0–100); flood_bam_* already prefixed + use return-period in name
    'flood_hazard_idx_norm':       'flood_hazard_historic_idx',
    'flood_hazard_fut_idx_norm':   'flood_hazard_midcentury_idx',
    # Drought sub-components and composite indices (0–100)
    'Dr_delta_JA_max_pre':         'drought_delta_temp_ja_historic',
    'Dr_delta_JA_max_fut':         'drought_delta_temp_ja_midcentury',
    # drought_delta_spei12_midcentury already correctly named
    'drought_hazard_idx_norm':     'drought_hazard_historic_idx',
    'drought_hazard_fut_idx_norm': 'drought_hazard_midcentury_idx',
    # Fire
    'fhsz':                        'fire_fhsz',
    'fhsz_responsibility':         'fire_fhsz_responsibility',
    'wui_type':                    'fire_wui_type',
}
output = output.rename(columns=rename_map)

# Explicit column order
facility_cols = [
    'facilityid', 'name', 'address', 'city', 'state', 'zip', 'telephone',
    'type', 'status', 'population', 'county', 'countyfips', 'country',
    'website', 'securelvl', 'capacity', 'geometry',
    'longitude', 'latitude', 'tract_geoid', 'capacity_percent',
    'dist_nearest_medical_mi', 'in_urban_area_2020',
]
heat_cols = [
    'heat_uhi_normalized',
    'heat_days_over_90_historic', 'heat_days_over_90_midcentury', 'heat_days_over_90_delta',
    'heat_hotnights_historic_pct', 'heat_hotnights_midcentury_pct',
    'heat_aqi_pctile',
    'heat_hazard_historic_idx', 'heat_hazard_midcentury_idx',
]
flood_cols = [
    'flood_bam_100_pct', 'flood_bam_500_pct',
    'flood_hazard_historic_idx', 'flood_hazard_midcentury_idx',
]
drought_cols = [c for c in [
    'drought_delta_temp_ja_historic', 'drought_delta_temp_ja_midcentury',
    'drought_delta_spei12_midcentury',
    'drought_hazard_historic_idx', 'drought_hazard_midcentury_idx',
] if c in output.columns]
fire_cols = ['fire_fhsz', 'fire_fhsz_responsibility', 'fire_wui_type']

output = output[facility_cols + heat_cols + flood_cols + drought_cols + fire_cols]

output.to_csv('data/allfacilities_climate_hazards.csv', index=False)
print(f'Saved {len(output)} rows × {len(output.columns)} columns to data/allfacilities_climate_hazards.csv')
print(f'\nColumns:')
for c in output.columns:
    print(f'  {c}')

Saved 357 rows × 44 columns to data/allfacilities_climate_hazards.csv

Columns:
  facilityid
  name
  address
  city
  state
  zip
  telephone
  type
  status
  population
  county
  countyfips
  country
  website
  securelvl
  capacity
  geometry
  longitude
  latitude
  tract_geoid
  capacity_percent
  dist_nearest_medical_mi
  in_urban_area_2020
  heat_uhi_normalized
  heat_days_over_90_historic
  heat_days_over_90_midcentury
  heat_days_over_90_delta
  heat_hotnights_historic_pct
  heat_hotnights_midcentury_pct
  heat_aqi_pctile
  heat_hazard_historic_idx
  heat_hazard_midcentury_idx
  flood_bam_100_pct
  flood_bam_500_pct
  flood_hazard_historic_idx
  flood_hazard_midcentury_idx
  drought_delta_temp_ja_historic
  drought_delta_temp_ja_midcentury
  drought_delta_spei12_midcentury
  drought_hazard_historic_idx
  drought_hazard_midcentury_idx
  fire_fhsz
  fire_fhsz_responsibility
  fire_wui_type
